# Predictive Analytics: Support Vector Machines with Regression for Census Tract

Approach for SVM:
– Simply start without a kernel. Then, gradually make your model complex by integrating different
kind of kernels. Also, use grid search to find optimal values for your hyperparameters.
– How good is your model? Evaluate your model’s performance and comment on its shortfalls.
– Show how you model’s performance varies as you increase or decrease temporal or spatial
resolution How does your performance change when you only use census tract as spatial units?
– How could the model be improved further? Explain some of the improvement levers that you might
focus on in a follow-up project.

In [69]:
from run_config import PATHS

In [70]:
#TODO: embedding ansehen -> vielleicht austauschen
#TODO: make more time efficient
#TODO: change to thundersvm -> installation umstädlich

In [ ]:
GRID_SAMPLE = 100_000 # if validation set over GRID_SEARCH use only GRID_SEARCH rows of data due to runtime issues, for grid search
SPATIAL_ENCODING = "latlong" # options: embedding, latlong, onehot
MODE = "full" # options: full, sample
TIME_UNIT = "4H" # options: 1H, 4H, 24H

In [72]:
CENSUS_PATH = "../data/full/raw_data/Census_Tracts.csv"
COMM_PATH = "../data/full/raw_data/Community_Areas.csv"

In [ ]:
import pandas as pd
import numpy as np
import matplotlib as plt
import datetime
from sklearn.svm import SVR 
from sklearn.svm import LinearSVR
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import RandomizedSearchCV 
# explicitly require this experimental feature
from sklearn.experimental import enable_halving_search_cv # noqa
# now you can import normally from model_selection
from sklearn.model_selection import HalvingGridSearchCV
from sklearn.compose import TransformedTargetRegressor
from sklearn.pipeline import Pipeline
from sklearn.kernel_approximation import Nystroem
from joblib import load, dump
from joblib import Memory
import h3

from shapely import wkt

import geopandas as gpd
from shapely.geometry import Polygon
from srai.neighbourhoods import H3Neighbourhood
from srai.loaders import OSMOnlineLoader
from srai.joiners import IntersectionJoiner
from srai.h3 import ring_buffer_h3_regions_gdf
from srai.embedders import Hex2VecEmbedder

import networkx as nx
from libpysal.weights import Queen
from node2vec import Node2Vec

## Preparations

In [74]:
INPUT = PATHS.train_test_dir

In [75]:
SPATIAL_UNIT = "CENSUS_TRACTS"

# Paths, depending on spatial and time unit
DATA_PATH_TRAIN = INPUT / f"GOLD_{TIME_UNIT}_DEMAND_{SPATIAL_UNIT}_TRAIN.parquet"
DATA_PATH_TEST = INPUT / f"GOLD_{TIME_UNIT}_DEMAND_{SPATIAL_UNIT}_TEST.parquet"
DATA_PATH_VAL = INPUT / f"GOLD_{TIME_UNIT}_DEMAND_{SPATIAL_UNIT}_VAL.parquet"



MODEL_PATH = "../models"

# Target and feature selection
TARGET_COL = "trip_count"
EXCLUDE_COLS = [
    TARGET_COL,
    "datetime_hour",      # real timestamp would lead to much leakage
    "_split_bucket",      # only used for splitting
    "month", # cyclic feature is used instead
    "weekday", # cyclic feature is used instead
    "hour", # cyclic feature is used instead
    # all other taxi data columns must be excluded to prevent leakage
    "trip_seconds_sum",
    "trip_seconds_mean",
    "trip_seconds_min",
    "trip_seconds_max",
    "trip_miles_sum",
    "trip_miles_mean",
    "trip_miles_min",
    "trip_miles_max",
    "fare_sum",
    "fare_mean",
    "fare_min",
    "fare_max",
    "tips_sum",
    "tips_mean",
    "tips_min",
    "tips_max",
    "tolls_sum",
    "tolls_mean",
    "tolls_min",
    "tolls_max",
    "extras_sum",
    "extras_mean",
    "extras_min",
    "extras_max",
    "trip_total_sum",
    "trip_total_mean",
    "trip_total_min",
    "trip_total_max",
    "most_common_payment_type",
    "h3_cell", # spatial units are getting encoded
    "census_tract",
    "community_area",
    "lat",
    "lon",
    "date",
    "h3_resolution",
]

Load data and select features and target

In [76]:
# Load data
train_df = pd.read_parquet(DATA_PATH_TRAIN)
test_df = pd.read_parquet(DATA_PATH_TEST)
val_df = pd.read_parquet(DATA_PATH_VAL)

In [81]:
train_df.head()

,datetime_hour,month,weekday,hour,month_sin,month_cos,weekday_sin,weekday_cos,hour_sin,hour_cos,...,tolls_max,extras_sum,extras_mean,extras_min,extras_max,trip_total_sum,trip_total_mean,trip_total_min,trip_total_max,most_common_payment_type
0,2026-03-15 12:00:00,3,7,12,0.866025,0.500000,-0.781831,0.623490,1.224647e-16,-1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,No trips
1,2025-02-03 12:00:00,2,1,12,0.500000,0.866025,0.000000,1.000000,1.224647e-16,-1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,No trips
2,2025-02-03 12:00:00,2,1,12,0.500000,0.866025,0.000000,1.000000,1.224647e-16,-1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,No trips
3,2025-02-08 04:00:00,2,6,4,0.500000,0.866025,-0.974928,-0.222521,8.660254e-01,0.5,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,No trips
4,2026-03-15 12:00:00,3,7,12,0.866025,0.500000,-0.781831,0.623490,1.224647e-16,-1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,No trips


In [82]:
# Create X and y
def feature_cols(train_df):
    feature_cols = [
        col for col in train_df.columns
        if col not in EXCLUDE_COLS
    ]
    return feature_cols

Spatial Encoding: LatLong

In [83]:
def spherical_encode(lat, lon):
    lat_rad = np.radians(lat)
    lon_rad = np.radians(lon)
    x = np.cos(lat_rad) * np.cos(lon_rad)
    y = np.cos(lat_rad) * np.sin(lon_rad)
    z = np.sin(lat_rad)
    return np.stack([x, y, z], axis=-1)

In [84]:
train_df

,datetime_hour,month,weekday,hour,month_sin,month_cos,weekday_sin,weekday_cos,hour_sin,hour_cos,...,tolls_max,extras_sum,extras_mean,extras_min,extras_max,trip_total_sum,trip_total_mean,trip_total_min,trip_total_max,most_common_payment_type
0,2026-03-15 12:00:00,3,7,12,0.866025,5.000000e-01,-0.781831,0.623490,1.224647e-16,-1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,No trips
1,2025-02-03 12:00:00,2,1,12,0.500000,8.660254e-01,0.000000,1.000000,1.224647e-16,-1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,No trips
2,2025-02-03 12:00:00,2,1,12,0.500000,8.660254e-01,0.000000,1.000000,1.224647e-16,-1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,No trips
3,2025-02-08 04:00:00,2,6,4,0.500000,8.660254e-01,-0.974928,-0.222521,8.660254e-01,0.5,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,No trips
4,2026-03-15 12:00:00,3,7,12,0.866025,5.000000e-01,-0.781831,0.623490,1.224647e-16,-1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,No trips
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1785847,2026-01-08 04:00:00,1,4,4,0.000000,1.000000e+00,0.433884,-0.900969,8.660254e-01,0.5,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,No trips
1785848,2025-01-21 20:00:00,1,2,20,0.000000,1.000000e+00,0.781831,0.623490,-8.660254e-01,0.5,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,No trips
1785849,2025-04-06 00:00:00,4,7,0,1.000000,6.123234e-17,-0.781831,0.623490,0.000000e+00,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,No trips
1785850,2025-10-05 04:00:00,10,7,4,-1.000000,-1.836970e-16,-0.781831,0.623490,8.660254e-01,0.5,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,No trips


In [85]:
# encode into lat long
if (SPATIAL_ENCODING == "latlong"):
    print("Encoding: latlong and Unit: census_tract")
       
    # load census tract
    census_data = pd.read_csv(CENSUS_PATH, dtype={"CENSUS_T_1": str})
    census_data["CENSUS_T_1"] = census_data["CENSUS_T_1"].str.zfill(11)

    tract_centroids = census_data.set_index("CENSUS_T_1")[["TRACT_CE_3", "TRACT_CE_2"]]
    tract_centroids.columns = ["lat", "lon"]

    for df in (train_df, val_df, test_df):
        df["census_tract"] = df["census_tract"].astype(str).str.zfill(11)
        df["lat"] = df["census_tract"].map(tract_centroids["lat"])
        df["lon"] = df["census_tract"].map(tract_centroids["lon"])

        # sanity check, catch silent join failures early
        n_missing = df["lat"].isna().sum()
        if n_missing:
            print(f"Warning: {n_missing}/{len(df)} rows failed to match a tract centroid")
        n_missing = df["lon"].isna().sum()
        if n_missing:
            print(f"Warning: {n_missing}/{len(df)} rows failed to match a tract centroid")

Encoding: latlong and Unit: census_tract


In [86]:
train_df

,datetime_hour,month,weekday,hour,month_sin,month_cos,weekday_sin,weekday_cos,hour_sin,hour_cos,...,extras_mean,extras_min,extras_max,trip_total_sum,trip_total_mean,trip_total_min,trip_total_max,most_common_payment_type,lat,lon
0,2026-03-15 12:00:00,3,7,12,0.866025,5.000000e-01,-0.781831,0.623490,1.224647e-16,-1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,No trips,41.906639,-87.689580
1,2025-02-03 12:00:00,2,1,12,0.500000,8.660254e-01,0.000000,1.000000,1.224647e-16,-1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,No trips,41.816264,-87.718927
2,2025-02-03 12:00:00,2,1,12,0.500000,8.660254e-01,0.000000,1.000000,1.224647e-16,-1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,No trips,41.867850,-87.698385
3,2025-02-08 04:00:00,2,6,4,0.500000,8.660254e-01,-0.974928,-0.222521,8.660254e-01,0.5,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,No trips,41.867850,-87.698385
4,2026-03-15 12:00:00,3,7,12,0.866025,5.000000e-01,-0.781831,0.623490,1.224647e-16,-1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,No trips,41.862661,-87.712875
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1785847,2026-01-08 04:00:00,1,4,4,0.000000,1.000000e+00,0.433884,-0.900969,8.660254e-01,0.5,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,No trips,41.847434,-87.721944
1785848,2025-01-21 20:00:00,1,2,20,0.000000,1.000000e+00,0.781831,0.623490,-8.660254e-01,0.5,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,No trips,41.885494,-87.708693
1785849,2025-04-06 00:00:00,4,7,0,1.000000,6.123234e-17,-0.781831,0.623490,0.000000e+00,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,No trips,41.804893,-87.672166
1785850,2025-10-05 04:00:00,10,7,4,-1.000000,-1.836970e-16,-0.781831,0.623490,8.660254e-01,0.5,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,No trips,41.941312,-87.685616


In [87]:
if SPATIAL_ENCODING == "latlong":
    for df in (train_df, val_df, test_df):
        result = spherical_encode(df["lat"], df["lon"])  
        df["x"], df["y"], df["z"] = result.T  

    # add x, y, z 
    feature_cols = feature_cols(train_df)

    X_train = train_df[feature_cols]
    X_test = test_df[feature_cols]

    if len(val_df) > GRID_SAMPLE:
        val_df_grid = val_df.sample(n=GRID_SAMPLE, random_state=42)
    else:
        val_df_grid = val_df
    X_val_grid = val_df_grid[feature_cols]

### Spatial Encoding: Spatial Embedding

In [88]:
if (SPATIAL_ENCODING == "embedding"):
    census_data = gpd.read_file(CENSUS_PATH)
    census_data["geometry"] = gpd.GeoSeries.from_wkt(census_data["the_geom"])

    gdf = gpd.GeoDataFrame(census_data, geometry="geometry", crs="EPSG:4326")

    gdf["TRACT_FIPS"] = gdf["TRACT_FIPS"].astype(str)
    gdf = gdf.set_index("TRACT_FIPS")

    gdf = gdf.to_crs(epsg=5070)
    gdf = gdf[gdf.geometry.notnull() & gdf.geometry.is_valid]
    gdf = gdf[~gdf.index.duplicated(keep="first")]

    w = Queen.from_dataframe(gdf, use_index=True)
    G = w.to_networkx()
    print(f"Graph: {G.number_of_nodes()} tracts, {G.number_of_edges()} adjacency edges")

    node2vec = Node2Vec(
        G,
        dimensions=32,
        walk_length=20,
        num_walks=100,
        workers=4,
        p=1,
        q=1,
    )

    model = node2vec.fit(window=10, min_count=1, batch_words=4)

    embedding_dict = {node: model.wv[node] for node in G.nodes()}
    emb_df = pd.DataFrame.from_dict(embedding_dict, orient="index")

    emb_cols = [f"emb_{i}" for i in range(emb_df.shape[1])]
    emb_df.columns = emb_cols

    train_df = train_df.merge(emb_df, left_on="census_tract", right_index=True, how="left")
    val_df = val_df.merge(emb_df, left_on="census_tract", right_index=True, how="left")
    test_df = test_df.merge(emb_df, left_on="census_tract", right_index=True, how="left")

    feature_cols = feature_cols + emb_cols

    X_train = train_df[feature_cols]
    X_val = val_df[feature_cols]
    X_test = test_df[feature_cols]

    val_df_grid = val_df.sample(n=GRID_SAMPLE, random_state=42)
    X_val_grid = val_df_grid[feature_cols]


Create y

In [89]:
y_train = train_df[TARGET_COL]
y_test = test_df[TARGET_COL]
y_val_grid = val_df_grid[TARGET_COL]

### Scale

In [90]:
X_val_grid

,month_sin,month_cos,weekday_sin,weekday_cos,hour_sin,hour_cos,is_holiday,weather_station_distance_km,food_drink,landmark,...,skyc1_SCT,skyc1_BKN,skyc1_OVC,skyc1_VV,weather_station_MDW,weather_station_ORD,weather_station_IGQ,x,y,z
114391,0.866025,5.000000e-01,-0.433884,-0.900969,-8.660254e-01,0.5,0,6.464436,0.0,0.0,...,0,0,1,0,1,0,0,0.030239,-0.745167,0.666191
185233,0.000000,1.000000e+00,0.433884,-0.900969,-8.660254e-01,0.5,0,6.464512,4.0,0.0,...,0,0,1,0,1,0,0,0.030080,-0.744665,0.666760
49658,0.000000,1.000000e+00,-0.974928,-0.222521,8.660254e-01,-0.5,0,14.121123,1.0,1.0,...,0,0,0,0,1,0,0,0.029074,-0.743594,0.667999
84925,-0.500000,8.660254e-01,-0.974928,-0.222521,1.224647e-16,-1.0,0,12.593248,6.0,0.0,...,0,1,0,0,0,1,0,0.028796,-0.743264,0.668379
339841,0.866025,5.000000e-01,-0.433884,-0.900969,-8.660254e-01,-0.5,0,17.670639,1.0,0.0,...,0,1,0,0,1,0,0,0.030101,-0.743285,0.668297
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
205491,0.500000,-8.660254e-01,0.000000,1.000000,8.660254e-01,-0.5,0,10.330817,0.0,1.0,...,0,0,0,0,1,0,0,0.029970,-0.744083,0.667415
289813,-0.500000,-8.660254e-01,-0.433884,-0.900969,8.660254e-01,-0.5,0,17.125203,10.0,0.0,...,0,0,0,0,1,0,0,0.029804,-0.743298,0.668296
357672,0.500000,8.660254e-01,0.974928,-0.222521,-8.660254e-01,-0.5,0,9.917547,0.0,0.0,...,0,1,0,0,0,1,0,0.028299,-0.742735,0.668987
224167,0.500000,8.660254e-01,-0.974928,-0.222521,0.000000e+00,1.0,0,16.662652,9.0,2.0,...,0,0,1,0,1,0,0,0.029984,-0.743375,0.668203


In [91]:
X_train

,month_sin,month_cos,weekday_sin,weekday_cos,hour_sin,hour_cos,is_holiday,weather_station_distance_km,food_drink,landmark,...,skyc1_SCT,skyc1_BKN,skyc1_OVC,skyc1_VV,weather_station_MDW,weather_station_ORD,weather_station_IGQ,x,y,z
0,0.866025,5.000000e-01,-0.781831,0.623490,1.224647e-16,-1.0,0,14.388325,10.0,3.0,...,0,0,1,0,1,0,0,0.030003,-0.743629,0.667919
1,0.500000,8.660254e-01,0.000000,1.000000,1.224647e-16,-1.0,0,4.361592,2.0,0.0,...,0,0,1,0,1,0,0,0.029664,-0.744696,0.666744
2,0.500000,8.660254e-01,0.000000,1.000000,1.224647e-16,-1.0,0,10.142263,0.0,0.0,...,0,0,1,0,1,0,0,0.029907,-0.744085,0.667415
3,0.500000,8.660254e-01,-0.974928,-0.222521,8.660254e-01,0.5,0,10.142263,0.0,0.0,...,0,0,1,0,1,0,0,0.029907,-0.744085,0.667415
4,0.866025,5.000000e-01,-0.781831,0.623490,1.224647e-16,-1.0,0,9.131898,3.0,0.0,...,0,0,1,0,1,0,0,0.029721,-0.744153,0.667347
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1785847,0.000000,1.000000e+00,0.433884,-0.900969,8.660254e-01,0.5,0,7.282519,1.0,0.0,...,0,0,1,0,1,0,0,0.029610,-0.744335,0.667149
1785848,0.000000,1.000000e+00,0.781831,0.623490,-8.660254e-01,0.5,0,11.640736,1.0,1.0,...,0,0,0,0,1,0,0,0.029764,-0.743885,0.667644
1785849,1.000000,6.123234e-17,-0.781831,0.623490,0.000000e+00,1.0,0,6.975201,1.0,0.0,...,0,0,1,0,1,0,0,0.030277,-0.744804,0.666596
1785850,-1.000000,-1.836970e-16,-0.781831,0.623490,8.660254e-01,0.5,0,18.133806,4.0,0.0,...,0,0,0,0,1,0,0,0.030038,-0.743223,0.668369


### Grid Search

In [93]:
model = SVR()

In [ ]:
memory = Memory(location="/tmp/sklearn_cache", verbose=0)

# pipelines
pipe_linear = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', LinearSVR(max_iter=100_000,tol=1e-2))
])

pipe_kernel = Pipeline([
    ('scaler', StandardScaler()),
    ('feature_map', Nystroem()),
    ('svm', SVR(max_iter=50_000, tol=1e-2))
], memory=memory)

# regressor
ttr_linear = TransformedTargetRegressor(regressor=pipe_linear, transformer=StandardScaler())
ttr_kernel = TransformedTargetRegressor(regressor=pipe_kernel, transformer=StandardScaler())

# parameter for each kernel # excluded C=100, C=10, 0,001 excluded via testing due to convergance issues
param_grid_linear = {
    "regressor__svm__C": [1, 10, 30, 100], # 4h: , 1h: 24h: 
    "regressor__svm__epsilon": [0.01, 0.05, 0.1, 0.3],
}

param_grid_rbf_sigmoid = {
    "regressor__svm__C": [1, 10],
    "regressor__svm__epsilon": [0.01, 0.05, 0.1],
    "regressor__feature_map__kernel": ["rbf", "sigmoid"],
    "regressor__feature_map__gamma": [0.01, 0.1, 1],
    "regressor__feature_map__n_components": [100, 300, 500],
}

param_grid_poly = {
    "regressor__svm__C": [1, 10],
    "regressor__svm__epsilon": [0.01, 0.05, 0.1],
    "regressor__feature_map__kernel": ["poly"],
    "regressor__feature_map__degree": [3, 4],
    "regressor__feature_map__gamma": [0.01, 0.1, 1],
    "regressor__feature_map__n_components": [100, 300, 500],
}

grids = {}
configs = [
    ("linear", ttr_linear, param_grid_linear),
    ("rbf_sigmoid", ttr_kernel, param_grid_rbf_sigmoid),
    ("poly", ttr_kernel, param_grid_poly),
]

# doing gridsearch on all
for name, pipe, grid in configs:
    search = HalvingGridSearchCV(
        estimator=pipe,
        param_grid=grid,
        cv=2, # changed to 2 due to runtime issues
        scoring="r2",
        n_jobs=-1,
        error_score="raise"
    )
    search.fit(X_val_grid, y_val_grid)
    grids[name] = search
    print(name, "best score:", search.best_score_, "best params:", search.best_params_)

best_name = max(grids, key=lambda name: grids[name].best_score_)
grid_search = grids[best_name]
print("Overall best:", best_name, grid_search.best_params_)

linear best score: 0.3120808394605238 best params: {'regressor__svm__epsilon': 0.1, 'regressor__svm__C': 1}


KeyboardInterrupt: 

In [ ]:
# Best parameters
print("Best parameters:", grid_search.best_params_)
print("Best CV score:", grid_search.best_score_)

Best parameters: {'C': 1, 'degree': 3, 'epsilon': 1, 'gamma': 'scale', 'kernel': 'poly'}
Best CV score: 0.3961679353349734


### Train Model

In [ ]:
best_model = grid_search.best_estimator_

In [ ]:
# Train SVR 

best_model.fit(X_train, y_train)

,"kernel kernel: {'linear', 'poly', 'rbf', 'sigmoid', 'precomputed'} or callable, default='rbf'Specifies the kernel type to be used in the algorithm.If none is given, 'rbf' will be used. If a callable is given it isused to precompute the kernel matrix.For an intuitive visualization of different kernel typessee :ref:`sphx_glr_auto_examples_svm_plot_svm_regression.py`",'poly'
,"gamma gamma: {'scale', 'auto'} or float, default='scale'Kernel coefficient for 'rbf', 'poly' and 'sigmoid'.- if ``gamma='scale'`` (default) is passed then it uses 1 / (n_features * X.var()) as value of gamma,- if 'auto', uses 1 / n_features- if float, must be non-negative... versionchanged:: 0.22 The default value of ``gamma`` changed from 'auto' to 'scale'.",0.1
,"epsilon epsilon: float, default=0.1Epsilon in the epsilon-SVR model. It specifies the epsilon-tubewithin which no penalty is associated in the training loss functionwith points predicted within a distance epsilon from the actualvalue. Must be non-negative.",1
,"degree degree: int, default=3Degree of the polynomial kernel function ('poly').Must be non-negative. Ignored by all other kernels.",3
,"coef0 coef0: float, default=0.0Independent term in kernel function.It is only significant in 'poly' and 'sigmoid'.",0.0
,"tol tol: float, default=1e-3Tolerance for stopping criterion.",0.001
,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive.The penalty is a squared l2. For an intuitive visualization of theeffects of scaling the regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",1
,"shrinking shrinking: bool, default=TrueWhether to use the shrinking heuristic.See the :ref:`User Guide <shrinking_svm>`.",True
,"cache_size cache_size: float, default=200Specify the size of the kernel cache (in MB).",200
,"verbose verbose: bool, default=FalseEnable verbose output. Note that this setting takes advantage of aper-process runtime setting in libsvm that, if enabled, may not workproperly in a multithreaded context.",False
,"max_iter max_iter: int, default=-1Hard limit on iterations within solver, or -1 for no limit.",-1


In [ ]:
# Make prediction 
y_pred = best_model.predict(X_test)

In [ ]:
y_pred

array([ 6.06623197,  1.34785082,  0.92840365, ..., -3.8927612 ,
       -0.86106653, -1.00588888])

In [ ]:
# Evaluation metrics

print("MAE:", mean_absolute_error(y_test, y_pred))
print("MSE:", mean_squared_error(y_test, y_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))
print("R2 Score:", r2_score(y_test, y_pred))

MAE: 16.578284127035385
MSE: 4093.5482175240877
RMSE: 63.98084258216742
R2 Score: 0.790151368685712


In [ ]:
# save model
dump(best_model, "../models/svm/model_" + SPATIAL_UNIT + "_" + TIME_UNIT + "_svr.joblib")
dump(grid_search, "../models/svm/grid_" + SPATIAL_UNIT +  "_" + TIME_UNIT + "_svr.joblib")

['../models/grid_community_svr.joblib']